# M2P1 Pre-Processing Tools
Pre-processing tools will be discussed and explained in the context of RNA-Seq. We will cover the following: 
- Quality assessment
- Adapter and low-quality base trimming/filtering

Below is an example RNA-Seq pipeline from start to finish:


*[Third-party RNA-seq workflow figure omitted from the public version. A similar figure is in the [HBC Intro-to-DGE lessons](https://hbctraining.github.io/Intro-to-DGE/).]*
Image from Slides produced by Richard H. Scheuermann, PhD J. Craig Venter Institute

**The steps we will cover in this notebook are preprocessing steps, including the quality control check on the initial fastq files, trimming, and quality control on the trimmed fastq files.** This prepares the output from Illumina Sequencing, our fastq files, for alignment (or pseudo-alignment) to a genome or transcriptome.

## My samples have been sequenced, now what to do with the FASTQ files?

Pre-processing of sequencing results is important in any NGS protocol. These experiments generate **millions** of short reads in FASTQ files, and before mapping reads to a genome or quantifying expression, they must ensure these reads are of high quality. There are usually 3 steps to Pre-Processing: 

1. Quality assessment
2. Adapter and low-quality base trimming
3. Filtering reads
4. Quality assessment of the trimmed reads

Raw reads contain adapter sequences, the synthetic oligos that are used in library prep, which need to be removed before analysis. They also contain low-quality bases and can be incomplete or too short. Illumina sequencing usually produces 100bp reads in the FASTQ files. 

In the event the quality of FASTQ files is not assessed and filtered/trimmed, reads can incorrectly map, leading to biased quantification and issues in differential analysis or splicing detection. 

## Quality control

The quality of reads should be assessed before and after trimming. There are many ways to do this, but common tools include [`FastQC`](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/) and [`fastp`](https://github.com/OpenGene/fastp). These tools generate plots displaying: 
- Per-base quality scores
- GC content
- Adapter content
- Sequence length distribution

In the case of `fastp`, trimming and quality are handled in a single command. This means each sample will generate before and after quality assessments and be trimmed within the same program. 

Students must be able to assess the quality of reads from the outputs of QC tools. The output of `fastQC` contains an HTML file, which can be viewed locally within a browser window to determine quality. The image below is an example of an output plot regarding a poor-quality read using Illumina sequencing. 

![M1P2_fastpQualityExample](https://genomics.sschmeier.com/_images/fastqc_bad1.png)
![M1P2_fastpQualityExampleGC](https://genomics.sschmeier.com/_images/fastqc_bad3.png)

This is an example of a bad read from the [`fastQC` tutorial by S. Schmeier](https://genomics.sschmeier.com/introduction).

*Images: Sebastian Schmeier, [Computational Genomics Tutorial](https://genomics.sschmeier.com/) ([sschmeier/genomics](https://github.com/sschmeier/genomics)), MIT License, Copyright (c) 2020 Sebastian Schmeier; unchanged.*

The quality scores of this read drop significantly over the course of 20+ base pairs. The Y-axis of this chart represents the **Phred**(*Q*) score, which logarithmically represents the probability that a base is called incorrectly. See the table below for the interpretation of **Phred** scores:
| Phred Score (Q) | Probability of error | Interpretation       |
| --------------- | -------------------- | -------------------- |
| 40              | 0.01%                | Very high confidence |
| 30              | 0.1%                 | High confidence      |
| 20              | 1%                   | Moderate confidence  |
| 10              | 10%                  | Low confidence       |

Secondly, when compared to the expected GC content(the number of GC bases relative to the number of AT bases), there is a large difference in the GC content seen within the read.  

**<u>Example Reports from the [FastQC Website](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/)</u>**

 - [Good Illumina Data](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/good_sequence_short_fastqc.html)
 - [Bad Illumina Data](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/bad_sequence_fastqc.html)
 - [Adapter dimer contaminated run](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/RNA-Seq_fastqc.html)
 - [Small RNA with read-through adapter](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/small_rna_fastqc.html)
 - [Reduced Representation BS-Seq](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/RRBS_fastqc.html)


**FURTHER LEARNING** [The MU Genomics core has a great write-up on the analysis of FASTQC outputs, and it is recommended to read through this](https://mugenomicscore.missouri.edu/PDF/FastQC_Manual.pdf)


## Trimming

Trimming removes unwanted bases from reads. This occurs in 3 main ways: 
- Adapter trimming: Remove sequences derived from library adapters.

- Quality trimming: Remove bases from the ends of reads with low quality scores.

- Length filtering: Discard reads that are too short after trimming.

The trimming software, either [`trimmomatic`](https://github.com/usadellab/Trimmomatic) or [`fastp`](https://github.com/OpenGene/fastp), scans the reads from both ends, removing bases below the quality threshold, detected adapter sequences, and reads shorter than a set minimum length. 

# Filtering Reads

Even after trimming some reads may be problematic due to their length, a large amount of ambiguity (too many `N's`), or they may be duplicate reads that will bias downstream analysis. This filtering improves mapping accuracy and quantification reliability by removing these low confidence reads. 

# Fastp: an all-in-one tool

`fastp` can be used to assess quality and trim/filter reads in a single command. This program can be ran on the command line, and will be later looped so that you can apply the tool to multiple files with a simple bash script. The inputs and outputs of fastp are shown below: 

1. Inputs

fastp primarily takes FASTQ files as input. These can be:

- Single-end reads: one FASTQ file per sample.

- Paired-end reads: two FASTQ files per sample (R1 and R2).

Parameters can be added to fastp to adjust trimming with more specificity: 

| Option             | Meaning                                             |
| ------------------ | --------------------------------------------------- |
| `-i`               | Input FASTQ file for R1 (forward reads)             |
| `-I`               | Input FASTQ file for R2 (reverse reads, paired-end) |
| `-o`               | Output trimmed FASTQ for R1                         |
| `-O`               | Output trimmed FASTQ for R2                         |
| `-h`               | Generate an HTML QC report                          |
| `-j`               | Generate a JSON QC report                           |
| `-q 20`            | Trim bases with quality < 20                        |
| `-l 20`            | Discard reads shorter than 20 bp after trimming     |
| `--n_base_limit 5` | Discard reads with >5 ambiguous bases               |
| `-w 4`             | Use 4 CPU threads for faster processing             |


2. Outputs of `fastp`:

- Trimmed FASTQ files

   - Single-end: one output FASTQ file (-o).

  - Paired-end: two output FASTQ files (-o and -O) for R1 and R2.

- QC Reports (optional but **HIGHLY recommended**)

    - HTML report (-h) → interactive summary with plots. Contains both pre-/post-trimming QC analytics.

    - JSON report (-j) → structured data for programmatic use.

**Note:** Filtered reads are removed by trimming/quality/length/N-base filters and are excluded from trimmed output `FASTQ` files.

Below is an example of a fastp command: 

```
fastp \
  -i sample_R1.fastq \
  -I sample_R2.fastq \
  -o sample_R1_trimmed.fastq \
  -O sample_R2_trimmed.fastq \
  -h sample_fastp_report.html \
  -j sample_fastp_report.json \
  -q 20 \
  -l 20 \
  --n_base_limit 5 \
  -w 4
```